# Statistika podatkov za diplomo (Colab)

Zvezek **ne spreminja nobenih podatkov**. Iz datotek na Drive izračuna številke za poglavji Materiali in metode ter Rezultati in jih shrani kot majhne datoteke v `MyDrive/Diploma/diploma_statistika/`:

| Datoteka | Za kaj |
|---|---|
| `verzije.json` | verzije Pythona, paketov in GPU (razdelek 3.8) |
| `tabela1_celice_po_bolnikih.csv` | tabela 1: celice po bolniku in kvadrantu, kloni, ekspandirane celice, merilo izbire |
| `povzetek_podatkov.json` | skupna števila celic, klonov, CD4/CD8 |
| `qc_po_datotekah.csv` | koliko celic odstrani vsak korak kontrole kakovosti (po datoteki .h5) |
| `qc_hist_geni.csv`, `qc_hist_mito.csv` | podatki za sliko 5 (porazdelitev števila genov in deleža mitohondrijskih prepisov) |
| `fluksi_povzetek.json` | osnovne lastnosti fluksov scFEA po zagonih (razdelek 3.5 in 4.3) |
| `fluksi_ponovljivost.csv` | ujemanje fluksov med zagoni scFEA (razdelek 4.3) |
| `fluksi_moduli_ekspanzija.csv` | razlike v modulih med ekspandiranimi in neekspandiranimi celicami (slika 8) |
| `fluksi_kosi_sestava.csv`, `fluksi_kosi_eta2.csv` | sestava kosov po 5000 celic, ki jih je scFEA obdelal ločeno, in delež variance modulov, ki ga pojasni kos oziroma bolnik |
| `scfea_moduli.csv`, `scfea_geni.json` | opis modulov scFEA in koliko naših genov je v modulih (razdelek 3.5) |
| `umap_bolnik23.csv` | koordinate UMAP resničnih in generiranih celic izpuščenega bolnika (slika 7) |

**Kako poženeš:** Runtime → Run all. Za razdelek 3 (surove datoteke .h5) izberi High-RAM, ker naloži vse surove matrike kot zvezek 01. Če želiš samo tabelo 1, nastavi `RUN_QC_RAW = False`. GPU ni potreben.

Na koncu mi datoteke iz mape `diploma_statistika` pošlji v niti (so majhne).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, sys, glob, re, platform, importlib
import numpy as np
import pandas as pd

# Poti enake kot v notebooks/01_preprocessing.ipynb
RAW_DIR  = '/content/drive/MyDrive/Diploma/data/GSE200996_RAW'
META_DIR = '/content/drive/MyDrive/Diploma/data'
PROC_DIR = '/content/drive/MyDrive/Diploma/data/processed'
OUT_DIR  = '/content/drive/MyDrive/Diploma/diploma_statistika'

MIN_EXP     = 20     # merilo izbire izpuščenih bolnikov (notebooks_flux/00_diagnostika_pacientov.ipynb)
MIN_GENES   = 200    # kontrola kakovosti (01_preprocessing)
MAX_MITO    = 20     # %
RUN_QC_RAW  = True   # razdelek 3: naloži surove .h5 (High-RAM)

os.makedirs(OUT_DIR, exist_ok=True)
print('Izhod:', OUT_DIR)

## 1. Verzije programske opreme

In [ ]:
def ver(mod):
    try:
        m = importlib.import_module(mod)
        return getattr(m, '__version__', 'ni podatka')
    except Exception as e:
        return 'ni nameščen'

paketi = ['numpy', 'pandas', 'scipy', 'scanpy', 'anndata', 'torch', 'sklearn', 'umap', 'matplotlib', 'seaborn', 'cobra']
verzije = {'python': platform.python_version(), 'platforma': platform.platform()}
verzije.update({p: ver(p) for p in paketi})
try:
    import torch
    verzije['cuda'] = torch.version.cuda
    verzije['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ni GPU'
except Exception:
    verzije['gpu'] = 'ni podatka'
with open(os.path.join(OUT_DIR, 'verzije.json'), 'w') as f:
    json.dump(verzije, f, indent=2, ensure_ascii=False)
print(json.dumps(verzije, indent=2, ensure_ascii=False))

## 2. Tabela 1: celice po bolniku in kvadrantu

Kvadrant = tkivo (kri, tumor) × faza (pred, po zdravljenju). Ekspanzija je definirana enako kot pri vrednotenju: klon je ekspandiran, če ima v krvi po zdravljenju več celic kot pred njim (`df_all_tcrs`, stolpca 1 in 0). `bolnik_indeks` je številka, ki jo uporabljajo logi (`heldout_patient`), `bolnik_id` pa oznaka iz GEO.

In [ ]:
labels     = pd.read_pickle(os.path.join(PROC_DIR, 'data_labels.pkl'))
labels_str = pd.read_pickle(os.path.join(PROC_DIR, 'data_labels_str.pkl'))
tcrs       = pd.read_pickle(os.path.join(PROC_DIR, 'df_all_tcrs.pkl'))

assert len(labels) == len(labels_str), 'data_labels in data_labels_str nista poravnana'
L = labels.reset_index(drop=True)
S = labels_str.reset_index(drop=True)

tissue  = L['Tissue'].astype(int).values            # 0 kri, 1 tumor
stage   = L['Treatment Stage'].astype(int).values   # 0 pred, 1 po
patient = L['Patient'].astype(int).values
ctype   = L['SubCellType'].astype(int).values       # 0 CD4, 1 CD8
tcr_idx = L['CDR3(Beta1)'].astype(int).values

blood_expanded = (tcrs.iloc[:, 1].fillna(0) > tcrs.iloc[:, 0].fillna(0)).values

rows = []
for pid in sorted(np.unique(patient)):
    m = patient == pid
    ids = S.loc[m, 'Patient'].unique()
    r = {'bolnik_indeks': int(pid), 'bolnik_id': ','.join(map(str, ids)), 'celic': int(m.sum()),
         'CD4': int((m & (ctype == 0)).sum()), 'CD8': int((m & (ctype == 1)).sum())}
    for t, tn in [(0, 'kri'), (1, 'tumor')]:
        for s, sn in [(0, 'pred'), (1, 'po')]:
            r[f'{tn}_{sn}'] = int((m & (tissue == t) & (stage == s)).sum())
    r['klonov'] = int(len(np.unique(tcr_idx[m])))
    bp = m & (tissue == 0) & (stage == 0)
    r['ekspandiranih_celic_kri_pred'] = int(blood_expanded[tcr_idx[bp]].sum()) if bp.any() else 0
    r['ekspandiranih_klonov_kri_pred'] = int(len(np.unique(tcr_idx[bp][blood_expanded[tcr_idx[bp]]]))) if bp.any() else 0
    r['izpolnjuje_merilo'] = r['ekspandiranih_celic_kri_pred'] >= MIN_EXP
    rows.append(r)

tab1 = pd.DataFrame(rows)
tab1.to_csv(os.path.join(OUT_DIR, 'tabela1_celice_po_bolnikih.csv'), index=False)

povzetek = {
    'celic': int(len(L)), 'bolnikov': int(len(tab1)),
    'CD4': int((ctype == 0).sum()), 'CD8': int((ctype == 1).sum()),
    'razlicnih_CDR3b': int(len(np.unique(tcr_idx))), 'vrstic_df_all_tcrs': int(len(tcrs)),
    'kri_pred': int(((tissue == 0) & (stage == 0)).sum()), 'kri_po': int(((tissue == 0) & (stage == 1)).sum()),
    'tumor_pred': int(((tissue == 1) & (stage == 0)).sum()), 'tumor_po': int(((tissue == 1) & (stage == 1)).sum()),
    'bolnikov_ki_izpolnjujejo_merilo': int(tab1['izpolnjuje_merilo'].sum()),
    'merilo': f'najmanj {MIN_EXP} ekspandiranih celic v krvi pred zdravljenjem',
}
with open(os.path.join(OUT_DIR, 'povzetek_podatkov.json'), 'w') as f:
    json.dump(povzetek, f, indent=2, ensure_ascii=False)

pd.set_option('display.max_rows', 100, 'display.width', 200)
print(json.dumps(povzetek, indent=2, ensure_ascii=False))
print(tab1.sort_values('ekspandiranih_celic_kri_pred', ascending=False).to_string(index=False))

## 3. Kontrola kakovosti na surovih datotekah .h5 (za sliko 5 in Metode)

Ponovi korake iz `01_preprocessing` (najmanj 200 genov, ujemanje s celicami iz metapodatkov, delež mitohondrijskih prepisov pod 20 %) in prešteje, koliko celic odstrani vsak korak. Za histograme uporabi celice, ki se ujemajo z metapodatki, **pred** filtriranjem po številu genov.

In [ ]:
if RUN_QC_RAW:
    import scanpy as sc, anndata

    def load_meta(fp):
        return pd.read_csv(fp, sep='\t', index_col=0)
    parts = []
    for fn, cls, tis in [('GSE200996_CD4.PBMC.single.cell.meta.data.txt', 'CD4', 'Blood'),
                         ('GSE200996_CD8.PBMC.single.cell.meta.data.txt', 'CD8', 'Blood'),
                         ('GSE200996_CD4.tumor.single.cell.meta.data.txt', 'CD4', 'Tumor'),
                         ('GSE200996_CD8.tumor.single.cell.meta.data.txt', 'CD8', 'Tumor')]:
        d = load_meta(os.path.join(META_DIR, fn)); d['CellClass'] = cls; d['Tissue_str'] = tis
        parts.append(d)
    meta_all = pd.concat(parts)
    meta_all = meta_all[meta_all['Stage'] != 'B3']
    meta_all['barcode_clean'] = meta_all.index.str.split('_').str[0]
    print('Celic v metapodatkih (brez B3):', len(meta_all))

    def patients_from_filename(fn):
        mt = re.search(r'(P\d{2}(?:-P\d{2})*)', os.path.basename(fn))
        return mt.group(1).split('-') if mt else None

    qc_rows, genes_all, mito_all = [], [], []
    for pattern, tis in [('*_GEX_sc_PBMC.h5', 'kri'), ('*_GEX_sc_tumor.h5', 'tumor')]:
        for f in sorted(glob.glob(os.path.join(RAW_DIR, pattern))):
            pats = patients_from_filename(f)
            if pats is None:
                continue
            valid_bc = set(meta_all.loc[meta_all['Patient_ID'].isin(pats), 'barcode_clean'])
            ad = sc.read_10x_h5(f); ad.var_names_make_unique()
            X = ad.X.tocsr()
            n_genes = np.asarray((X > 0).sum(axis=1)).ravel()
            total = np.asarray(X.sum(axis=1)).ravel()
            mt = ad.var_names.str.upper().str.startswith('MT-')
            mito_pct = 100 * np.asarray(X[:, np.where(mt)[0]].sum(axis=1)).ravel() / np.maximum(total, 1)
            bc = ad.obs_names.str.replace(r'-\d+$', '', regex=True)
            in_meta = np.asarray(bc.isin(valid_bc))
            ok_genes = n_genes >= MIN_GENES
            ok_mito = mito_pct < MAX_MITO
            qc_rows.append({'datoteka': os.path.basename(f), 'tkivo': tis, 'bolniki': '-'.join(pats),
                            'surove_crtne_kode': int(len(n_genes)),
                            'genov_vsaj_200': int(ok_genes.sum()),
                            'po_ujemanju_z_metapodatki': int((ok_genes & in_meta).sum()),
                            'po_mito_pod_20': int((ok_genes & in_meta & ok_mito).sum()),
                            'odstranjenih_zaradi_mito': int((ok_genes & in_meta & ~ok_mito).sum())})
            genes_all.append(n_genes[in_meta]); mito_all.append(mito_pct[in_meta & ok_genes])
            print(qc_rows[-1])
            del ad, X

    qc = pd.DataFrame(qc_rows)
    qc.to_csv(os.path.join(OUT_DIR, 'qc_po_datotekah.csv'), index=False)
    print(qc.sum(numeric_only=True))

    g = np.concatenate(genes_all); mpc = np.concatenate(mito_all)
    edges_g = np.linspace(0, 4.5, 91)                       # log10(genov + 1)
    hg, _ = np.histogram(np.log10(g + 1), bins=edges_g)
    pd.DataFrame({'log10_genov_od': edges_g[:-1], 'log10_genov_do': edges_g[1:], 'celic': hg}).to_csv(
        os.path.join(OUT_DIR, 'qc_hist_geni.csv'), index=False)
    edges_m = np.arange(0, 101, 1.0)
    hm, _ = np.histogram(np.clip(mpc, 0, 100), bins=edges_m)
    pd.DataFrame({'mito_od': edges_m[:-1], 'mito_do': edges_m[1:], 'celic': hm}).to_csv(
        os.path.join(OUT_DIR, 'qc_hist_mito.csv'), index=False)
    print('Celic, ujemajočih se z metapodatki (pred QC):', len(g), '| pod 200 genov:', int((g < MIN_GENES).sum()),
          '| mediana mito % (po pragu genov):', round(float(np.median(mpc)), 2))
else:
    print('Razdelek 3 preskočen (RUN_QC_RAW = False).')

## 4. Fluksi scFEA: lastnosti, ponovljivost in razlike med ekspandiranimi celicami

Prebere `flux_runs/data_flux_seed{0,1,2}.pkl` (izhod `notebooks_flux/01_scfea_flux.ipynb`). Za vsak zagon izračuna osnovne lastnosti, nato ujemanje med zagoni (Pearsonova korelacija po modulih čez celice in po celicah čez module) in za vsak modul primerjavo med celicami v krvi pred zdravljenjem, katerih klon se po zdravljenju razširi, in ostalimi. Fluksi so standardizirani z enako kot v `03_trim_flux` (po modulih, na vseh celicah). `auc_modula` je ploščina pod krivuljo ROC, če bi ekspanzijo napovedovali samo s tem modulom (0,5 = brez informacije).

In [ ]:
from scipy.stats import mannwhitneyu

FLUX_RUNS = os.path.join(PROC_DIR, 'flux_runs')
mod_names = [str(m) for m in pd.read_pickle(os.path.join(FLUX_RUNS, 'flux_module_names.pkl'))]
fluksi = {}
for s in [0, 1, 2]:
    fp = os.path.join(FLUX_RUNS, f'data_flux_seed{s}.pkl')
    if os.path.exists(fp):
        V = np.asarray(pd.read_pickle(fp), dtype=np.float64)
        assert V.shape == (len(L), len(mod_names)), f'seme {s}: oblika {V.shape} se ne ujema'
        fluksi[s] = V
print('Zagoni scFEA (semena):', list(fluksi), '| modulov:', len(mod_names))
assert fluksi, 'Ni datotek flux_runs/data_flux_seed*.pkl'

fpovz = {'modulov': len(mod_names), 'semena': list(fluksi), 'celic': int(len(L))}
for s, V in fluksi.items():
    sd = V.std(axis=0)
    d = {'delez_tocnih_nicel': round(float((V == 0).mean()), 4), 'mrtvih_modulov': int((sd < 1e-9).sum()),
         'povprecje': float(V.mean()), 'mediana': float(np.median(V)), 'min': float(V.min()), 'max': float(V.max())}
    meta = os.path.join(PROC_DIR, 'flux_batches', f'seed{s}', 'run_meta.json')
    if os.path.exists(meta):
        d['run_meta'] = json.load(open(meta))
    fpovz[f'seme{s}'] = d
with open(os.path.join(OUT_DIR, 'fluksi_povzetek.json'), 'w') as f:
    json.dump(fpovz, f, indent=2, ensure_ascii=False)
print(json.dumps(fpovz, indent=2, ensure_ascii=False))

def zstd(A, axis):
    return (A - A.mean(axis=axis, keepdims=True)) / (A.std(axis=axis, keepdims=True) + 1e-12)

rng = np.random.default_rng(0)
idx = rng.choice(len(L), min(20000, len(L)), replace=False)
pon = []
sem = sorted(fluksi)
for i in range(len(sem)):
    for j in range(i + 1, len(sem)):
        A, B = fluksi[sem[i]], fluksi[sem[j]]
        live = (A.std(0) > 1e-9) & (B.std(0) > 1e-9)
        r_mod = (zstd(A[:, live], 0) * zstd(B[:, live], 0)).mean(0)
        r_cel = (zstd(A[idx], 1) * zstd(B[idx], 1)).mean(1)
        pon.append({'seme_a': sem[i], 'seme_b': sem[j], 'modulov': int(live.sum()),
                    'r_moduli_mediana': float(np.median(r_mod)), 'r_moduli_q25': float(np.quantile(r_mod, .25)),
                    'r_moduli_q75': float(np.quantile(r_mod, .75)), 'r_moduli_min': float(r_mod.min()),
                    'delez_modulov_r_nad_0_9': float((r_mod > 0.9).mean()),
                    'r_celice_mediana': float(np.median(r_cel)), 'r_celice_q25': float(np.quantile(r_cel, .25)),
                    'r_celice_q75': float(np.quantile(r_cel, .75)),
                    'enaki_zagoni': bool(np.allclose(A, B))})
pon = pd.DataFrame(pon)
pon.to_csv(os.path.join(OUT_DIR, 'fluksi_ponovljivost.csv'), index=False)
print(pon.to_string(index=False))

def bh(p):
    p = np.asarray(p); n = len(p); o = np.argsort(p)
    q = p[o] * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n); out[o] = np.minimum(q, 1); return out

kri_pred = (tissue == 0) & (stage == 0)
eksp = blood_expanded[tcr_idx]
skupine = [('vsi_bolniki', kri_pred), ('bolnik_23', kri_pred & (patient == 23)), ('bolnik_24', kri_pred & (patient == 24))]
vrst = []
for s, V in fluksi.items():
    Z = (V - V.mean(axis=0)) / (V.std(axis=0) + 1e-8)      # kot v 03_trim_flux
    for ime, msk in skupine:
        e, n = msk & eksp, msk & ~eksp
        if e.sum() == 0 or n.sum() == 0:
            continue
        rows_s = []
        for k, mod in enumerate(mod_names):
            if V[:, k].std() < 1e-9:
                continue
            u, p = mannwhitneyu(Z[e, k], Z[n, k], alternative='two-sided')
            rows_s.append({'seme': s, 'skupina': ime, 'modul': mod, 'n_eksp': int(e.sum()), 'n_neeksp': int(n.sum()),
                           'z_eksp': float(Z[e, k].mean()), 'z_neeksp': float(Z[n, k].mean()),
                           'razlika_z': float(Z[e, k].mean() - Z[n, k].mean()),
                           'auc_modula': float(u / (e.sum() * n.sum())), 'p': float(p)})
        q = bh([r['p'] for r in rows_s])
        for r, qq in zip(rows_s, q):
            r['q_bh'] = float(qq)
        vrst += rows_s
mods = pd.DataFrame(vrst)
mods.to_csv(os.path.join(OUT_DIR, 'fluksi_moduli_ekspanzija.csv'), index=False)
top = mods[(mods.seme == min(fluksi)) & (mods.skupina == 'vsi_bolniki')].assign(a=lambda d: d.razlika_z.abs()).sort_values('a', ascending=False)
print(top.head(15).drop(columns='a').to_string(index=False))
print('Modulov z q < 0,05 (vsi bolniki, seme', min(fluksi), '):', int((top.q_bh < 0.05).sum()))

# Kosi: scFEA je vsak kos po CHUNK celic (v vrstnem redu data_labels) obdelal z ločeno naučeno mrežo
s0 = min(fluksi)
CHUNK = fpovz[f'seme{s0}'].get('run_meta', {}).get('CHUNK', 5000)
kos = np.arange(len(L)) // CHUNK
kv = np.array([f"{'kri' if t == 0 else 'tumor'}_{'pred' if g == 0 else 'po'}" for t, g in zip(tissue, stage)])
sest = pd.crosstab(kos, kv)
sest['bolnikov'] = pd.Series(patient).groupby(kos).nunique().values
sest['bolniki'] = pd.Series(patient).groupby(kos).apply(lambda x: ','.join(map(str, sorted(set(x))))).values
sest.index.name = 'kos'
sest.to_csv(os.path.join(OUT_DIR, 'fluksi_kosi_sestava.csv'))
print(sest.to_string())

def eta2(Z, g):
    mu = Z.mean(0); tot = ((Z - mu) ** 2).sum(0)
    btw = sum(((Z[g == k].mean(0) - mu) ** 2) * (g == k).sum() for k in np.unique(g))
    return btw / np.maximum(tot, 1e-12)
Z0 = fluksi[s0]; live = Z0.std(0) > 1e-9
eta = pd.DataFrame({'modul': np.array(mod_names)[live],
                    'eta2_kos': eta2(Z0[:, live], kos), 'eta2_bolnik': eta2(Z0[:, live], patient),
                    'eta2_kvadrant': eta2(Z0[:, live], kv)})
kp = kri_pred
eta['eta2_kos_kri_pred'] = eta2(Z0[kp][:, live], kos[kp])
eta['eta2_bolnik_kri_pred'] = eta2(Z0[kp][:, live], patient[kp])
eta.to_csv(os.path.join(OUT_DIR, 'fluksi_kosi_eta2.csv'), index=False)
print(eta.describe().loc[['mean', '50%', 'max']].to_string())

# Moduli scFEA in prekrivanje z našimi geni (datoteke iz repozitorija scFEA, kot v 01_scfea_flux)
try:
    if not os.path.exists('/content/scFEA/data'):
        import subprocess; subprocess.run(['git', 'clone', '-q', 'https://github.com/changwn/scFEA.git', '/content/scFEA'], check=True)
    mg = pd.read_csv('/content/scFEA/data/module_gene_m168.csv', index_col=0)
    module_genes = set(str(v).strip() for c in mg.columns for v in mg[c].dropna() if str(v).strip().lower() != 'nan')
    gene_names = [str(g) for g in pd.read_pickle(os.path.join(PROC_DIR, 'gene_names.pkl'))]
    cm = pd.read_csv('/content/scFEA/data/cmMat_c70_m168.csv', index_col=0)
    geni = {'genov_v_podatkih': len(gene_names), 'genov_v_modulih': len(module_genes),
            'nasih_genov_v_modulih': int(sum(g in module_genes for g in gene_names)),
            'modulov_v_tabeli_genov': int(mg.shape[0]), 'stehiometricna_matrika': list(cm.shape)}
    json.dump(geni, open(os.path.join(OUT_DIR, 'scfea_geni.json'), 'w'), indent=2, ensure_ascii=False)
    print(geni)
    info = [f for f in os.listdir('/content/scFEA/data') if 'M168' in f and 'information' in f.lower()]
    print('Datoteke z opisom modulov:', info)
    if info:
        pd.read_csv(os.path.join('/content/scFEA/data', info[0])).to_csv(os.path.join(OUT_DIR, 'scfea_moduli.csv'), index=False)
except Exception as ex:
    print('Opis modulov scFEA preskočen:', ex)

## 5. UMAP resničnih in generiranih celic izpuščenega bolnika (slika 7)

Za zagon `rna_tcr` z izpuščenim bolnikom 23 (seme 0) prebere `preds.npz` in shranjeni UMAP iz `03_trim`, ter shrani koordinate resničnih in generiranih celic po kvadrantih. Če datotek ni, razdelek samo izpiše opozorilo.

In [ ]:
UMAP_BOLNIK, UMAP_SEME = 23, 0
try:
    import pickle
    try:
        import umap
    except ImportError:
        import subprocess; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'umap-learn'], check=True)
        import umap
    cand = [os.path.join(PROC_DIR, 'runs', f'P{UMAP_BOLNIK}', f'seed{UMAP_SEME}', 'rna_tcr', 'preds.npz')]
    cand += sorted(glob.glob(os.path.join(PROC_DIR, 'runs', f'P{UMAP_BOLNIK}', '*', 'rna_tcr', 'preds.npz')))
    pfile = next((c for c in cand if os.path.exists(c)), None)
    assert pfile is not None, f'ni preds.npz za bolnika {UMAP_BOLNIK}'
    print('preds:', pfile)
    preds = np.load(pfile)['preds_rna']
    Xp = np.asarray(pd.read_pickle(os.path.join(PROC_DIR, 'data_rna_pca.pkl')))
    reducer = pickle.load(open(os.path.join(PROC_DIR, 'umap_trained_rna.pkl'), 'rb'))
    m = patient == UMAP_BOLNIK
    assert preds.shape[0] == m.sum(), f'preds {preds.shape[0]} != celic bolnika {m.sum()}'
    e_real = reducer.transform(Xp[m]); e_gen = reducer.transform(preds)
    tk = np.where(tissue[m] == 0, 'kri', 'tumor'); fa = np.where(stage[m] == 0, 'pred', 'po')
    um = pd.concat([pd.DataFrame({'umap1': e_real[:, 0], 'umap2': e_real[:, 1], 'vir': 'resnicna', 'tkivo': tk, 'faza': fa}),
                    pd.DataFrame({'umap1': e_gen[:, 0], 'umap2': e_gen[:, 1], 'vir': 'generirana', 'tkivo': tk, 'faza': fa})])
    um.to_csv(os.path.join(OUT_DIR, f'umap_bolnik{UMAP_BOLNIK}.csv'), index=False)
    print(um.groupby(['vir', 'tkivo', 'faza']).size())
except Exception as ex:
    print('Razdelek 5 preskočen:', ex)

## 6. Kaj je shranjeno

In [ ]:
for fn in sorted(os.listdir(OUT_DIR)):
    print(f'{fn:40s} {os.path.getsize(os.path.join(OUT_DIR, fn)):>8d} B')